In [0]:
dbutils.widgets.text("Landing_timestamp","")
dbutils.widgets.text("Table_name","")
dbutils.widgets.text("Merge_keys","")
dbutils.widgets.text("Load_type","")

landing_timestamp=dbutils.widgets.get("Landing_timestamp")
table_name=dbutils.widgets.get("Table_name")
merge_keys=dbutils.widgets.get("Merge_keys")
load_type=dbutils.widgets.get("Load_type")

print("=" * 80)
print("STEP 1: Job parameters received")
print("=" * 80)
print(f"  Landing_timestamp : '{landing_timestamp}'")
print(f"  Table_name        : '{table_name}'")
print(f"  Merge_keys        : '{merge_keys}'")
print(f"  Load_type         : '{load_type}'")

In [0]:
source_table = f"caresyncadb.bronze.{table_name}"
source_filter = f"Landing_timestamp = '{landing_timestamp}'"

print("=" * 80)
print("STEP 2: Reading data from bronze layer")
print("=" * 80)
print(f"  Source table : {source_table}")
print(f"  Filter       : {source_filter}")
print("  Equivalent SQL:")
print(f"    SELECT * FROM {source_table} WHERE {source_filter}")

df = spark.read.table(source_table).filter(source_filter)

source_count = df.count()
print(f"  Rows read from bronze : {source_count}")
print(f"  Source columns ({len(df.columns)}) : {df.columns}")
if source_count == 0:
    print("  WARNING: No rows found for this Landing_timestamp. Check the parameter value.")

In [0]:
from pyspark.sql import functions as F
target_table = f"caresyncadb.silver.{table_name}"
normalized_load_type = (load_type or 'FULL').strip().upper()
merge_keys_list = [key.strip() for key in merge_keys.split(",") if key.strip()]

updated_df = df.withColumn("insert_timestamp",F.current_timestamp())\
.withColumn("update_timestamp",F.current_timestamp())

print("=" * 80)
print("STEP 3: Preparing data for silver layer")
print("=" * 80)
print(f"  Target table         : {target_table}")
print(f"  Raw Load_type        : '{load_type}'")
print(f"  Normalized Load_type : '{normalized_load_type}'" + ("  (defaulted to FULL because Load_type was empty)" if not (load_type or '').strip() else ""))
print(f"  Merge keys (parsed)  : {merge_keys_list}")
print("  Added audit columns  : insert_timestamp, update_timestamp (current_timestamp())")
print(f"  Final columns ({len(updated_df.columns)}) : {updated_df.columns}")

In [0]:
print("=" * 80)
print(f"STEP 4: Writing to silver layer using Load_type = {normalized_load_type}")
print("=" * 80)

if normalized_load_type == "FULL":
    print(f"  FULL load: overwriting {target_table} with the prepared data")
    print("  Equivalent SQL:")
    print(f"    CREATE OR REPLACE TABLE {target_table} AS SELECT * FROM <updated_df>")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print(f"  FULL load completed for {target_table}")
elif normalized_load_type == "APPEND":
    print(f"  APPEND load: appending rows to {target_table}")
    print("  Equivalent SQL:")
    print(f"    INSERT INTO {target_table} SELECT * FROM <updated_df>")
    updated_df.write.mode("append").saveAsTable(target_table)
    print(f"  APPEND load completed for {target_table}")
elif normalized_load_type == "MERGE":
    table_exists = spark.catalog.tableExists(target_table)
    print(f"  Target table exists? {table_exists}")
    if not table_exists:
        print(f"  Target table not found, so creating it with a full write (first load)")
        print("  Equivalent SQL:")
        print(f"    CREATE OR REPLACE TABLE {target_table} AS SELECT * FROM <updated_df>")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"  Initial load completed for {target_table}")
    else:
        if not merge_keys_list:
            print("  WARNING: Merge_keys is empty, so the MERGE condition will be invalid.")
        source_view = "silver_merge_source"
        updated_df.createOrReplaceTempView(source_view)
        print(f"  Registered temp view '{source_view}' as the MERGE source")
        merge_condition = " AND ".join([f"target.`{key}` = source.`{key}`" for key in merge_keys_list])
        update_assignments = ", ".join(  [ f"target.`{column}` = source.`{column}`"
                                     for column in updated_df.columns 
                                     if column!="insert_timestamp"
                                     ])
        insert_columns = ",".join([f"`{column}`" for column in updated_df.columns])
        insert_values = ",".join([f"source.`{column}`" for column in updated_df.columns])

        print(f"  Merge condition    : {merge_condition}")
        print(f"  Update assignments : {update_assignments}")
        print(f"  Insert columns     : {insert_columns}")
        print(f"  Insert values      : {insert_values}")

        merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING {source_view} AS source
            ON {merge_condition}
            WHEN MATCHED THEN UPDATE SET {update_assignments}
            WHEN NOT MATCHED THEN INSERT ({insert_columns}) VALUES ({insert_values})
        """
        print("  MERGE query to be executed:")
        print(merge_sql)

        merge_result = spark.sql(merge_sql)
        print("  MERGE completed. Result metrics:")
        merge_result.show(truncate=False)
else:
    print(f"  ERROR: Unsupported Load_type '{load_type}'. Expected FULL, APPEND or MERGE.")
    raise ValueError(f"Unsupported Load_type: {load_type}")
        
        

In [0]:
record_count = updated_df.count()

print("=" * 80)
print("STEP 5: Load summary")
print("=" * 80)
print(f"  Source table  : caresyncadb.bronze.{table_name}")
print(f"  Target table  : {target_table}")
print(f"  Load type     : {normalized_load_type}")
print(f"  Records processed : {record_count}")
print("  Exiting notebook and returning record count to the caller")

dbutils.notebook.exit(str(record_count))